In [57]:
%cd /kaggle/working
!rm -rf DGM-HMamba verified data

# ۱) کد
!cp -r /kaggle/input/models/zarrinmahdavi/dgm-hmamba-main/pytorch/default/1/DGM-HMamba-main DGM-HMamba
%cd DGM-HMamba
!ls requirements.txt scripts/demo_real.py class_embeddings/glove.6B.300d.json

# ۲) داده‌ها — ⬅️ اینجا تغییر کرد: با %env تعریف می‌شود
%env DS=/kaggle/input/datasets/zarrinmahdavi/dataset
!mkdir -p data/IndianPines data/PaviaU data/Salinas
!cp $DS/Indian_pines_corrected.mat $DS/Indian_pines_gt.mat data/IndianPines/
!cp $DS/PaviaU.mat $DS/PaviaU_gt.mat data/PaviaU/
!cp $DS/salinas_corrected.mat $DS/salinas_gt.mat data/Salinas/

# ۳) مدل‌های آموزش‌دیده
!cp -r /kaggle/input/datasets/zarrinmahdavi/verified-ip2pu-v verified
!ls verified/s5_m42/model_final.pt

# ۴) چک‌ها
!python -c "import torch; print('GPU:', torch.cuda.is_available())"
!python scripts/demo_forward.py
print("=== SETUP COMPLETE ===")

/kaggle/working
/kaggle/working/DGM-HMamba
class_embeddings/glove.6B.300d.json  requirements.txt  scripts/demo_real.py
env: DS=/kaggle/input/datasets/zarrinmahdavi/dataset
verified/s5_m42/model_final.pt
GPU: True
Device: cuda
[Stage 1] L_cls=1.7126  L_diff=1.2008  L_adv=1.3779  total=2.4508
[Stage 2] L_epi=1.0887  L_gcl=0.9589  total=0.2048  acc=50.00%
Trainable parameters: 0.307 M
Smoke test passed.
=== SETUP COMPLETE ===


In [58]:
runner = r'''
#!/bin/bash
cd /kaggle/working/DGM-HMamba
mkdir -p runs /kaggle/working/backup_zips
run_exp() {
  name=$1; shift
  if [ -f "runs/$name/done.flag" ]; then echo "SKIP $name"; return; fi
  mkdir -p runs/$name                                    # ← این خط اضافه شد
  echo "=== RUNNING: $name ==="
  python scripts/demo_real.py --data-root data "$@" --output-dir runs/$name > runs/$name/log.txt 2>&1
  if [ $? -eq 0 ]; then
    touch runs/$name/done.flag; echo "=== DONE: $name ==="
    zip -qr /kaggle/working/backup_zips/$name.zip runs/$name
    echo "=== BACKUP SAVED: $name.zip ==="
  else
    echo "=== FAILED: $name ==="
  fi
}
'''
with open("/kaggle/working/runner.sh","w") as f: f.write(runner)

exp = r'''
source /kaggle/working/runner.sh
run_exp s5_m42_eval --weights verified/s5_m42/model_final.pt
run_exp quick_train --stage1-iters 300 --stage2-steps 200
run_exp verified_s5_m42 --preset verified --split-seed 5 --seed 42
run_exp verified_glove_s5_m42 --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json
run_exp support_free_s5_m42 --preset verified --split-seed 5 --seed 42 --stage2-steps 0 --class-embeddings class_embeddings/glove.6B.300d.json --support-free
run_exp anchor_feature_s5_m42 --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json --anchor-init feature
run_exp anchor_none_s5_m42 --preset verified --split-seed 5 --seed 42 --anchor-init none
echo "ALL FINISHED"
'''
with open("/kaggle/working/session.sh","w") as f: f.write(exp)

import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)

Started! PID = 765


In [54]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)
print("First results (s5_m42_eval) in ~5 minutes.")

Started! PID = 669
First results (s5_m42_eval) in ~5 minutes.


In [55]:
!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/ 2>/dev/null

SKIP s5_m42_eval
SKIP quick_train
SKIP verified_s5_m42
=== RUNNING: verified_glove_s5_m42 ===
total 7.2M
-rw-r--r-- 1 root root 2.5M Nov 27 08:29 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 08:16 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.5M Nov 27 08:55 verified_s5_m42.zip


In [ ]:
!cat /kaggle/working/session.log



In [ ]:
!cat /kaggle/working/DGM-HMamba/runs/s5_m42_eval/demo_summary.json 2>/dev/null || unzip -p /kaggle/working/backup_zips/s5_m42_eval.zip runs/s5_m42_eval/demo_summary.json

!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/
!tail -3 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null

import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)

!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/

In [61]:
!tail -5 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null

        t_outputs, *args, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
    )  # Calls into the C++ engine to run the backward pass
    ^
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 154.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 84.81 MiB is free. Process 614 has 9.06 GiB memory in use. Process 630 has 3.91 GiB memory in use. Including non-PyTorch memory, this process has 1.51 GiB memory in use. Of the allocated memory 1.14 GiB is allocated by PyTorch, and 243.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


In [62]:
!pkill -f session.sh; pkill -f demo_real.py; sleep 3
!nvidia-smi          # باید memory usage را پایین (~0) نشان دهد

Fri Nov 27 09:20:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             69W /   70W |    9277MiB /  15360MiB |     93%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [63]:
!kill -9 614 630 2>/dev/null; sleep 2; nvidia-smi

Fri Nov 27 09:20:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             41W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [68]:
runner = r'''
#!/bin/bash
cd /kaggle/working/DGM-HMamba
mkdir -p runs /kaggle/working/backup_zips
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
run_exp() {
  name=$1; shift
  if [ -f "runs/$name/done.flag" ]; then echo "SKIP $name"; return; fi
  rm -rf runs/$name
  mkdir -p runs/$name
  echo "=== RUNNING: $name ==="
  python scripts/demo_real.py --data-root data --grad-checkpoint --map-batch 64 "$@" --output-dir runs/$name > runs/$name/log.txt 2>&1
  if [ $? -eq 0 ]; then
    touch runs/$name/done.flag; echo "=== DONE: $name ==="
    zip -qr /kaggle/working/backup_zips/$name.zip runs/$name
    echo "=== BACKUP SAVED: $name.zip ==="
  else
    echo "=== FAILED: $name ==="
  fi
}
'''
with open("/kaggle/working/runner.sh","w") as f: f.write(runner)
print("runner.sh fixed: --grad-checkpoint --map-batch 64")

runner.sh fixed: --grad-checkpoint --map-batch 64


In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

In [69]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

Restarted! PID = 993


In [ ]:
!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log | tail -8
!tail -5 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -5
!ls -lh /kaggle/working/backup_zips/
!tail -2 /kaggle/working/DGM-HMamba/runs/$(grep RUNNING /kaggle/working/session.log | tail -1 | awk '{print $3}')/log.txt 2>/dev/null

In [69]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

Restarted! PID = 993


In [ ]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

In [ ]:
import subprocess
# قدیمی‌ترین bash session.sh را پیدا و می‌کشیم (python را نمی‌کشیم)
out = subprocess.run(["bash","-c","ps -eo pid,etimes,cmd | grep 'bash /kaggle/working/session.sh' | grep -v grep | sort -k2 -n | head -1 | awk '{print $1}'"], capture_output=True, text=True).stdout.strip()
if out:
    subprocess.run(["kill", out]); print("killed duplicate bash PID:", out)
else:
    print("no duplicate")

In [73]:
print(1+1)

2


In [74]:
import subprocess
print(subprocess.run(["bash","-c","grep -E 'DONE|FAILED' /kaggle/working/session.log | tail -8; echo ---; ls /kaggle/working/backup_zips/"], capture_output=True, text=True).stdout)

=== FAILED: quick_train ===
=== FAILED: verified_s5_m42 ===
=== FAILED: verified_glove_s5_m42 ===
=== FAILED: support_free_s5_m42 ===
=== FAILED: anchor_feature_s5_m42 ===
=== FAILED: anchor_none_s5_m42 ===
=== DONE: s5_m42_eval ===
=== DONE: quick_train ===
---
quick_train.zip
s5_m42_eval.zip
verified_s5_m42.zip



In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Resumed! PID =", proc.pid)

!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

In [85]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

=== FAILED: support_free_s5_m42 ===
=== FAILED: anchor_feature_s5_m42 ===
=== FAILED: anchor_none_s5_m42 ===
=== DONE: s5_m42_eval ===
=== DONE: quick_train ===
=== DONE: verified_s5_m42 ===
=== RUNNING: verified_glove_s5_m42 ===
total 7.2M
-rw-r--r-- 1 root root 2.5M Nov 27 09:35 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 09:30 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.5M Nov 27 10:25 verified_s5_m42.zip


In [81]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/verified_s5_m42/demo_summary.json'))
e = s['episodic_metrics']
print("5-shot OA: %.2f  AA: %.2f" % (e['5way_5shot']['oa_mean']*100, e['5way_5shot']['aa_mean']*100))
print("1-shot OA: %.2f  AA: %.2f" % (e['5way_1shot']['oa_mean']*100, e['5way_1shot']['aa_mean']*100))
print("Map    OA: %.2f  AA: %.2f" % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100))

KeyError: 'aa_mean'

In [88]:
!grep -E "DONE|FAILED|RUNNING" /kaggle/working/session.log | tail -4
!ls -lh /kaggle/working/backup_zips/
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!tail -2 /kaggle/working/DGM-HMamba/runs/verified_glove_s5_m42/log.txt 2>/dev/null

=== DONE: verified_glove_s5_m42 ===
=== RUNNING: support_free_s5_m42 ===
=== DONE: support_free_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
total 12M
-rw-r--r-- 1 root root 2.5M Nov 27 09:35 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 09:30 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.4M Nov 27 11:40 support_free_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 11:29 verified_glove_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 10:59 verified_s5_m42.zip
100 %, 9805 MiB
0 %, 3 MiB
Classification map: 9-way, 5 labelled pixels per class drawn from P_t^te (10 independent draws); all other P_t^te pixels are queries ...
  encoding all 20509 pixels of the test region on cuda (the longest step on a CPU) ...


In [87]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/verified_s5_m42/demo_summary.json'))
e = s['episodic_metrics']
print("5-shot OA: %.2f  AA: %.2f  k: %.3f" % (e['5way_5shot']['oa_mean']*100, e['5way_5shot']['aa']*100, e['5way_5shot']['kappa']))
print("1-shot OA: %.2f  AA: %.2f  k: %.3f" % (e['5way_1shot']['oa_mean']*100, e['5way_1shot']['aa']*100, e['5way_1shot']['kappa']))
print("Map    OA: %.2f  AA: %.2f  k: %.3f" % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100, s['map_metrics']['kappa_mean']))

5-shot OA: 91.25  AA: 91.37  k: 0.901
1-shot OA: 87.17  AA: 87.22  k: 0.855
Map    OA: 82.59  AA: 85.09  k: 0.781


In [89]:
!grep -E "DONE|FAILED|RUNNING" /kaggle/working/session.log | tail -4
!ls -lh /kaggle/working/backup_zips/

=== DONE: support_free_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
=== DONE: verified_glove_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
total 12M
-rw-r--r-- 1 root root 2.5M Nov 27 09:35 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 09:30 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.4M Nov 27 11:40 support_free_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 12:03 verified_glove_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 10:59 verified_s5_m42.zip


In [90]:
!grep -E "DONE|FAILED|RUNNING|ALL FINISHED" /kaggle/working/session.log | tail -6
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

=== DONE: verified_glove_s5_m42 ===
=== RUNNING: support_free_s5_m42 ===
=== DONE: support_free_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
=== DONE: verified_glove_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
5
total 12M
-rw-r--r-- 1 root root 2.5M Nov 27 09:35 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 09:30 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.4M Nov 27 11:40 support_free_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 12:03 verified_glove_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 10:59 verified_s5_m42.zip


In [91]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/support_free_s5_m42/demo_summary.json'))
m = s.get('support_free') or {}
print('support-free enabled:', s['args'].get('support_free'))
print('Map OA: %.2f  AA: %.2f' % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100))
e = s['episodic_metrics']
print('5-shot OA: %.2f | 1-shot OA: %.2f' % (e['5way_5shot']['oa_mean']*100, e['5way_1shot']['oa_mean']*100))

support-free enabled: True
Map OA: 23.11  AA: 30.94
5-shot OA: 56.75 | 1-shot OA: 53.31


In [93]:
!grep -E "DONE|FAILED|RUNNING|ALL FINISHED" /kaggle/working/session.log | tail -4
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

=== DONE: support_free_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
=== DONE: verified_glove_s5_m42 ===
=== RUNNING: anchor_feature_s5_m42 ===
5
total 12M
-rw-r--r-- 1 root root 2.5M Nov 27 09:35 quick_train.zip
-rw-r--r-- 1 root root 2.3M Nov 27 09:30 s5_m42_eval.zip
-rw-r--r-- 1 root root 2.4M Nov 27 11:40 support_free_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 12:03 verified_glove_s5_m42.zip
-rw-r--r-- 1 root root 2.5M Nov 27 10:59 verified_s5_m42.zip
